0. IMPORTS

In [1]:
import os
import time
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
from datasets import Dataset
from datetime import timedelta
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback, 
    TrainerCallback
)

1. PATHS & CONFIGURATION

In [2]:
MODEL_NAME = "google-bert/bert-base-multilingual-cased" 
TRAIN_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\train.csv'
VAL_PATH = r'D:\Major Project\SpamX\machine_learning\dataset\validation.csv'
OUTPUT_DIR = r'D:\Major Project\SpamX\machine_learning\models\mBERT'

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

c:\Users\jeeve\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


2. ETA & PROGRESS LOGGING

In [4]:
class ETALoggingCallback(TrainerCallback):
    def __init__(self):
        self.start_time = None

    def on_train_begin(self, args, state, control, **kwargs):
        self.start_time = time.time()
        print(f"\nmBERT Finetuning Started | Total Steps: {state.max_steps}")

    def on_log(self, args, state, control, logs=None, **kwargs):
        if self.start_time and state.global_step > 0:
            elapsed = time.time() - self.start_time
            avg_time_per_step = elapsed / state.global_step
            remaining_steps = state.max_steps - state.global_step
            eta_seconds = remaining_steps * avg_time_per_step
            
            eta_str = str(timedelta(seconds=int(eta_seconds)))
            raw_loss = logs.get("loss", "N/A")
            loss_val = f"{raw_loss:.4f}" if isinstance(raw_loss, (float, int)) else "Calculating..."
            
            print(f"Step: {state.global_step}/{state.max_steps} | Loss: {loss_val} | ETA: {eta_str} | Progress: {(state.global_step/state.max_steps)*100:.2f}%")

3. SLIDING WINDOW

In [5]:
def tokenize_comment_only(examples):
    texts = [str(com) for com in examples["Comment"]]
    
    tokenized = tokenizer(
        texts,
        truncation=True,
        max_length=128, 
        stride=32,     
        return_overflowing_tokens=True,
        padding="max_length"
    )
    
    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    tokenized["labels"] = [examples["Label"][i] for i in sample_mapping]
    return tokenized

4. METRICS

In [6]:
class FocalLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        gamma, alpha = 2.0, 0.25 
        probs = F.softmax(logits, dim=-1)
        pt = probs.gather(1, labels.unsqueeze(1)).squeeze(1)
        
        loss = -alpha * (1 - pt) ** gamma * (pt + 1e-8).log()
        return (loss.mean(), outputs) if return_outputs else loss.mean()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions),
        "precision": precision_score(labels, predictions),
        "recall": recall_score(labels, predictions)
    }

5. DATA LOADING & PREPARATION

In [7]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

train_ds = Dataset.from_pandas(train_df).map(tokenize_comment_only, batched=True, remove_columns=train_df.columns.tolist())
val_ds = Dataset.from_pandas(val_df).map(tokenize_comment_only, batched=True, remove_columns=val_df.columns.tolist())

Map:   0%|          | 0/61645 [00:00<?, ? examples/s]

Map:   0%|          | 0/6438 [00:00<?, ? examples/s]

6. MODEL INITIALIZATION & FINETUNING

In [8]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

training_args = TrainingArguments(
    output_dir=os.path.join(OUTPUT_DIR, "checkpoints"),
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    logging_steps=50,
    evaluation_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True, 
    report_to="none",
    disable_tqdm=True
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2), ETALoggingCallback()]
)

trainer.train()

c:\Users\jeeve\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



mBERT Finetuning Started | Total Steps: 11754
Step: 50/11754 | Loss: 0.0182 | ETA: 0:34:27 | Progress: 0.43%
{'loss': 0.0182, 'grad_norm': 0.326610267162323, 'learning_rate': 1.9914922579547388e-05, 'epoch': 0.012761613067891782}
Step: 100/11754 | Loss: 0.0141 | ETA: 0:32:35 | Progress: 0.85%
{'loss': 0.0141, 'grad_norm': 0.2514212429523468, 'learning_rate': 1.9829845159094777e-05, 'epoch': 0.025523226135783564}
Step: 150/11754 | Loss: 0.0129 | ETA: 0:31:55 | Progress: 1.28%
{'loss': 0.0129, 'grad_norm': 0.28640204668045044, 'learning_rate': 1.9744767738642167e-05, 'epoch': 0.03828483920367534}
Step: 200/11754 | Loss: 0.0116 | ETA: 0:31:30 | Progress: 1.70%
{'loss': 0.0116, 'grad_norm': 0.5138330459594727, 'learning_rate': 1.9659690318189556e-05, 'epoch': 0.05104645227156713}
Step: 250/11754 | Loss: 0.0133 | ETA: 0:31:12 | Progress: 2.13%
{'loss': 0.0133, 'grad_norm': 0.3563467562198639, 'learning_rate': 1.9574612897736943e-05, 'epoch': 0.0638080653394589}
Step: 300/11754 | Loss: 0.01

TrainOutput(global_step=7500, training_loss=0.006565435975790024, metrics={'train_runtime': 1607.3361, 'train_samples_per_second': 116.983, 'train_steps_per_second': 7.313, 'train_loss': 0.006565435975790024, 'epoch': 1.9142419601837672})

7. FINAL SAVE

In [9]:
final_path = os.path.join(OUTPUT_DIR, "final_mbert")
trainer.save_model(final_path)
tokenizer.save_pretrained(final_path)

print(f"\nmBERT finetuning complete! Saved to: {final_path}")


mBERT finetuning complete! Saved to: D:\Major Project\SpamX\machine_learning\models\mBERT\final_mbert
